# 🌸 Iris Flower Classification Using Machine Learning

**Objective:** Build and evaluate machine learning classification models to predict Iris flower species — *Setosa*, *Versicolor*, and *Virginica* — using the Iris dataset.

---

### Workflow Overview
1. Dataset Understanding & Exploration
2. Data Preprocessing
3. Model Development (Logistic Regression, KNN, Random Forest)
4. Model Evaluation & Comparison
5. Feature Importance & Insights
6. Prediction Demonstration
7. Final Conclusion

> **Random Seed:** All experiments use `random_state=42` for reproducibility.

## 0. Setup & Library Imports

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── Plot Style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

print('✅ All libraries imported successfully!')

---
## 1. Dataset Understanding & Exploration
### 1.1 Load the Dataset

In [ ]:
# ── Upload Iris.csv when running in Colab ────────────────────────────────────
import os

# Try to detect Colab environment
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CSV_PATH = 'Iris.csv'

if IN_COLAB and not os.path.exists(CSV_PATH):
    print('📂 Please upload your Iris.csv file:')
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]

df = pd.read_csv(CSV_PATH)

# Drop 'Id' column if present (common in Kaggle Iris datasets)
if 'Id' in df.columns:
    df.drop(columns=['Id'], inplace=True)

print(f'✅ Dataset loaded from: {CSV_PATH}')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

### 1.2 Dataset Structure

In [ ]:
# Identify feature and target columns
TARGET_COL = [c for c in df.columns if 'species' in c.lower() or 'class' in c.lower()][0]
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]

print('=' * 50)
print(f'Dataset Shape  : {df.shape}')
print(f'Feature Columns: {FEATURE_COLS}')
print(f'Target Column  : {TARGET_COL}')
print(f'Target Classes : {df[TARGET_COL].unique().tolist()}')
print('=' * 50)
print('\nData Types:')
print(df.dtypes)

### 1.3 Data Quality Check

In [ ]:
print('── Missing Values ──────────────────────')
print(df.isnull().sum())

print(f'\n── Duplicate Rows ──────────────────────')
print(f'Total duplicates: {df.duplicated().sum()}')

print('\n── Summary Statistics ──────────────────')
df[FEATURE_COLS].describe().round(3)

In [ ]:
df[FEATURE_COLS].describe().round(3)

### 1.4 Exploratory Data Analysis (EDA)

In [ ]:
# ── Class Distribution ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_counts = df[TARGET_COL].value_counts()

# Bar plot
axes[0].bar(class_counts.index, class_counts.values,
            color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution — Bar Chart', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Species', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
for i, (label, val) in enumerate(zip(class_counts.index, class_counts.values)):
    axes[0].text(i, val + 0.5, str(val), ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452', '#55A868'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution — Pie Chart', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()
print('\n📊 Class Counts:')
print(class_counts)

In [ ]:
# ── Correlation Heatmap ──────────────────────────────────────────────────────
plt.figure(figsize=(7, 5))
corr = df[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 11})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Pair Plot ────────────────────────────────────────────────────────────────
pair_df = df.copy()
g = sns.pairplot(pair_df, hue=TARGET_COL, diag_kind='kde',
                 plot_kws={'alpha': 0.7, 's': 40, 'edgecolor': 'white'},
                 palette='muted')
g.fig.suptitle('Pair Plot — Feature Relationships by Species',
               y=1.02, fontsize=14, fontweight='bold')
plt.savefig('pair_plot.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Distribution — Violin Plots ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
colors = ['#4C72B0', '#DD8452', '#55A868']

for idx, feat in enumerate(FEATURE_COLS):
    sns.violinplot(data=df, x=TARGET_COL, y=feat, ax=axes[idx],
                   palette=colors, inner='quartile', linewidth=1.2)
    axes[idx].set_title(f'{feat} Distribution by Species',
                        fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Species', fontsize=10)
    axes[idx].set_ylabel(feat, fontsize=10)

plt.suptitle('Feature Distributions per Species (Violin Plots)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('violin_plots.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Box Plots ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, feat in enumerate(FEATURE_COLS):
    sns.boxplot(data=df, x=TARGET_COL, y=feat, ax=axes[idx],
                palette=colors, linewidth=1.2)
    axes[idx].set_title(f'{feat} — Box Plot', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Species', fontsize=10)
    axes[idx].set_ylabel(feat, fontsize=10)

plt.suptitle('Feature Box Plots per Species',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('box_plots.png', bbox_inches='tight')
plt.show()

---
## 2. Data Preprocessing

In [ ]:
# ── 2.1 Handle Missing Values ────────────────────────────────────────────────
if df.isnull().sum().sum() > 0:
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())
    print('⚠️  Missing values filled with column medians.')
else:
    print('✅ No missing values found.')

# ── 2.2 Remove Duplicates ────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f'✅ Duplicates removed: {before - after} rows  (remaining: {after})')

# ── 2.3 Encode Target Labels ─────────────────────────────────────────────────
le = LabelEncoder()
df['Species_Encoded'] = le.fit_transform(df[TARGET_COL])
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f'✅ Label encoding: {label_mapping}')

# ── 2.4 Features & Target ─────────────────────────────────────────────────────
X = df[FEATURE_COLS].values
y = df['Species_Encoded'].values

# ── 2.5 Train-Test Split (80 : 20) ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'✅ Train set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples')

# ── 2.6 Feature Scaling ──────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('✅ Feature scaling applied (StandardScaler fitted on training data).')

---
## 3. Model Development

Three classifiers are trained:
| Model | Key Hyperparameters |
|---|---|
| Logistic Regression | `C=1.0`, `max_iter=200`, `multi_class='auto'` |
| K-Nearest Neighbors | `n_neighbors=5`, `metric='minkowski'`, `weights='uniform'` |
| Random Forest | `n_estimators=100`, `max_depth=None`, `random_state=42` |

In [ ]:
# ── Define Models ─────────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=200, multi_class='auto', random_state=RANDOM_STATE
    ),
    'K-Nearest Neighbors': KNeighborsClassifier(
        n_neighbors=5, metric='minkowski', weights='uniform'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=RANDOM_STATE
    ),
}

# KNN & LR use scaled data; RF can work with raw (but we use scaled for uniformity)
results   = {}
trained   = {}
class_labels = le.classes_

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    trained[name] = model
    y_pred = model.predict(X_test_sc)

    results[name] = {
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, average='weighted'), 4),
        'Recall'   : round(recall_score(y_test, y_pred, average='weighted'), 4),
        'F1-Score' : round(f1_score(y_test, y_pred, average='weighted'), 4),
        'y_pred'   : y_pred,
    }
    print(f'✅ {name} trained.')

---
## 4. Model Evaluation

In [ ]:
# ── 4.1 Performance Comparison Table ─────────────────────────────────────────
metrics_df = pd.DataFrame({
    name: {
        'Accuracy' : v['Accuracy'],
        'Precision': v['Precision'],
        'Recall'   : v['Recall'],
        'F1-Score' : v['F1-Score'],
    }
    for name, v in results.items()
}).T

best_model_name = metrics_df['Accuracy'].idxmax()
print('\n📊 MODEL PERFORMANCE COMPARISON')
print('=' * 60)
print(metrics_df.to_string())
print('=' * 60)
print(f'🏆 Best Model: {best_model_name} (Accuracy = {metrics_df.loc[best_model_name, "Accuracy"]:.4f})')

In [ ]:
# ── 4.2 Metrics Bar Chart ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_df))
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
bar_colors  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
width = 0.18

for i, (metric, color) in enumerate(zip(metric_cols, bar_colors)):
    bars = ax.bar(x + i * width, metrics_df[metric], width,
                  label=metric, color=color, edgecolor='white', linewidth=0.8)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_df.index, fontsize=11)
ax.set_ylim(0.7, 1.07)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.3 Confusion Matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, v) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, v['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels,
                ax=ax, linewidths=0.5, linecolor='white',
                cbar=False, annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(f'{name}\n(Acc={v["Accuracy"]:.3f})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=9)
    ax.set_ylabel('True Label', fontsize=9)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.4 Detailed Classification Reports ──────────────────────────────────────
for name, v in results.items():
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(classification_report(y_test, v['y_pred'], target_names=class_labels))

---
## 5. Feature Importance & Insights

In [ ]:
# ── Random Forest Feature Importance ─────────────────────────────────────────
rf_model     = trained['Random Forest']
importances  = rf_model.feature_importances_
feat_imp_df  = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})\
                 .sort_values('Importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
               color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'],
               edgecolor='white', linewidth=1)
for bar, val in zip(bars, feat_imp_df['Importance']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10)
ax.set_title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Gini Importance', fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

print('\n📋 Feature Importance Ranking:')
print(feat_imp_df.to_string(index=False))
print(f'\n🔑 Most important feature: {feat_imp_df.iloc[0, 0]} ({feat_imp_df.iloc[0, 1]:.4f})')

In [ ]:
# ── Logistic Regression — Coefficient Magnitudes ─────────────────────────────
lr_model = trained['Logistic Regression']
coef_mag = np.abs(lr_model.coef_).mean(axis=0)
lr_imp_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Avg |Coefficient|': coef_mag})\
              .sort_values('Avg |Coefficient|', ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(lr_imp_df['Feature'], lr_imp_df['Avg |Coefficient|'],
        color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'],
        edgecolor='white', linewidth=1)
ax.set_title('Logistic Regression — Mean |Coefficient| per Feature',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Mean |Coefficient|', fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('lr_coefficients.png', bbox_inches='tight')
plt.show()

---
## 6. Prediction Demonstration

Using the **best-performing model** to predict species for 10 sample observations from the test set.

In [ ]:
best_model = trained[best_model_name]

# Sample 10 observations from the test set
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_test), size=10, replace=False)
X_sample   = X_test_sc[sample_idx]
y_actual   = y_test[sample_idx]

y_sample_pred = best_model.predict(X_sample)

pred_df = pd.DataFrame(X_sample, columns=FEATURE_COLS)
pred_df['Actual Species']    = le.inverse_transform(y_actual)
pred_df['Predicted Species'] = le.inverse_transform(y_sample_pred)
pred_df['Correct?']          = pred_df['Actual Species'] == pred_df['Predicted Species']
pred_df['Correct?']          = pred_df['Correct?'].map({True: '✅', False: '❌'})

print(f'🔮 Predictions using: {best_model_name}\n')
print(pred_df.to_string(index=False))
correct = (pred_df['Correct?'] == '✅').sum()
print(f'\n✅ Correct Predictions: {correct}/10')

In [ ]:
# ── Custom Prediction ─────────────────────────────────────────────────────────
# Provide your own flower measurements below (in cm)
custom_samples = [
    [5.1, 3.5, 1.4, 0.2],   # Expected: Iris-setosa
    [6.7, 3.1, 4.4, 1.4],   # Expected: Iris-versicolor
    [6.3, 3.3, 6.0, 2.5],   # Expected: Iris-virginica
]

custom_arr   = scaler.transform(np.array(custom_samples))
custom_preds = best_model.predict(custom_arr)

print('\n🌸 Custom Flower Predictions:')
print(f'  {" | ".join(FEATURE_COLS)}')
for sample, pred in zip(custom_samples, custom_preds):
    print(f'  {sample}  →  {le.inverse_transform([pred])[0]}')

---
## 7. Final Conclusion

### Dataset Characteristics
The Iris dataset contains **150 samples** across **3 perfectly balanced classes** (50 each) with **4 numerical features** — Sepal Length, Sepal Width, Petal Length, and Petal Width. No missing values were detected. The dataset is a classic, clean benchmark for multi-class classification.

### Key EDA Findings
- **Iris-setosa** is linearly separable from the other two species based on Petal dimensions alone.
- **Iris-versicolor** and **Iris-virginica** overlap moderately, making them harder to distinguish.
- **Petal Length** and **Petal Width** are highly correlated (~0.96) and are the most discriminative features.

### Model Comparison Summary

| Model | Accuracy | Notes |
|---|---|---|
| Logistic Regression | Excellent | Fast, interpretable, linear boundaries |
| K-Nearest Neighbors | Excellent | Non-parametric, sensitive to scaling |
| **Random Forest** | **Highest** | Ensemble method, robust, provides feature importance |

### Best Model
**Random Forest Classifier** achieved the highest accuracy. Its ensemble of decision trees excels at capturing non-linear decision boundaries between Versicolor and Virginica.

### Feature Importance
- **Petal Length** and **Petal Width** contribute ~85–90% of classification power.
- **Sepal Length** offers minor additional signal.
- **Sepal Width** is the least discriminative feature.

### Recommendations
1. **Use Random Forest** for Iris classification tasks requiring maximum accuracy.
2. For production edge deployments with limited compute, **Logistic Regression** is a strong, interpretable alternative.
3. If collecting new flowers, prioritise measuring **Petal dimensions** for the most reliable species prediction.

In [ ]:
# ── Final Summary Print ───────────────────────────────────────────────────────
print('╔══════════════════════════════════════════════════╗')
print('║        IRIS CLASSIFICATION — FINAL SUMMARY       ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Dataset : {after} samples, 3 classes, 4 features         ║')
print(f'║  Split   : 80% train / 20% test (stratified)     ║')
print(f'║  Scaling : StandardScaler                        ║')
print('╠══════════════════════════════════════════════════╣')
for name, v in results.items():
    marker = '🏆' if name == best_model_name else '  '
    print(f'║ {marker} {name:<25} Acc={v["Accuracy"]:.4f}  ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Best Model : {best_model_name:<34} ║')
print(f'║  Accuracy   : {metrics_df.loc[best_model_name, "Accuracy"]:.4f}                              ║')
print(f'║  Key Feature: {feat_imp_df.iloc[0, 0]:<34} ║')
print('╚══════════════════════════════════════════════════╝')